In [1]:
# ======================================================================
# CELL 1 — GLOBALS
# ======================================================================
import os, gc, torch, numpy as np, pandas as pd, cv2
from pathlib import Path
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import (Dataset, DataLoader, WeightedRandomSampler)
from sklearn.metrics import precision_recall_fscore_support
from torch.amp import GradScaler, autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

SEMANTIC_GROUPS = [
    'contamination','cut','deformation','fracture',
    'hole_void','minor_defect','scratch','surface_quality'
]
NUM_DEFECT_TYPES  = 8
NUM_PRODUCT_TYPES = 17

DEFECT_TYPE_TO_IDX = {n:i for i,n in enumerate(SEMANTIC_GROUPS)}
IDX_TO_DEFECT_TYPE = {i:n for i,n in enumerate(SEMANTIC_GROUPS)}

DEFECT_GROUP_MAP = {
    'crack':'fracture','fracture':'fracture','faulty_imprint':'fracture',
    'hole':'hole_void','void':'hole_void','pit':'hole_void',
    'blowhole':'hole_void','scratch':'scratch','score':'scratch',
    'stain':'surface_quality','color':'surface_quality',
    'rough':'surface_quality','uneven':'surface_quality',
    'inclusion':'surface_quality','discolor':'surface_quality',
    'pilling':'surface_quality','bent':'deformation',
    'bent_lead':'deformation','squeeze':'deformation',
    'deformation':'deformation','contamination':'contamination',
    'glue':'contamination','oil':'contamination',
    'glue_strip':'contamination','liquid':'contamination',
    'metal_contamination':'contamination','missing':'minor_defect',
    'misplaced':'minor_defect','flip':'minor_defect',
    'missing_hole':'minor_defect','thread':'minor_defect',
    'cable_swap':'minor_defect','combined':'minor_defect',
    'cut':'cut','hole_void':'hole_void',
    'surface_quality':'surface_quality','minor_defect':'minor_defect',
}

print(f"NUM_DEFECT_TYPES  : {NUM_DEFECT_TYPES}")
print(f"NUM_PRODUCT_TYPES : {NUM_PRODUCT_TYPES}")
print(f"SEMANTIC_GROUPS   : {SEMANTIC_GROUPS}")
print("✅ CELL 1 COMPLETE")

Device : cuda
NUM_DEFECT_TYPES  : 8
NUM_PRODUCT_TYPES : 17
SEMANTIC_GROUPS   : ['contamination', 'cut', 'deformation', 'fracture', 'hole_void', 'minor_defect', 'scratch', 'surface_quality']
✅ CELL 1 COMPLETE


In [2]:
# ======================================================================
# CELL 2 — CSV + LABEL PROCESSING
# ======================================================================
CSV_PATH = '/home/sufi/merged_dataset_metadata_augmented.csv'
df_3head = pd.read_csv(CSV_PATH)

if 'path'     in df_3head.columns:
    df_3head = df_3head.rename(columns={'path':'image_path'})
if 'category' in df_3head.columns:
    df_3head = df_3head.rename(columns={'category':'product_type'})

print(f"Loaded : {len(df_3head):,} rows")

def compute_binary(v):
    if pd.isna(v): return 0
    return 0 if str(v).strip().lower() in \
        ('0','good','normal','ok','casting_ok') else 1

def remap_defect(raw):
    if pd.isna(raw): return -1
    s = str(raw).strip().lower()
    if s in ('good','normal','casting_ok',''): return -1
    sem = DEFECT_GROUP_MAP.get(s)
    if sem is None and s in SEMANTIC_GROUPS: sem = s
    if sem is None:
        for k,v in DEFECT_GROUP_MAP.items():
            if k in s: sem=v; break
    return -1 if sem is None else DEFECT_TYPE_TO_IDX.get(sem,-1)

df_3head['binary_label']      = df_3head['label'].apply(compute_binary)
df_3head['defect_type_label'] = df_3head['defect_type'].apply(remap_defect)
df_3head.loc[df_3head['binary_label']==0,'defect_type_label'] = -1

all_products        = sorted(df_3head['product_type'].dropna()
                              .unique().tolist())
PRODUCT_TYPE_TO_IDX = {p:i for i,p in enumerate(all_products)}
IDX_TO_PRODUCT_TYPE = {i:p for i,p in enumerate(all_products)}
NUM_PRODUCT_TYPES   = len(all_products)

df_3head['product_type_label'] = [
    PRODUCT_TYPE_TO_IDX.get(str(v),0) if not pd.isna(v) else 0
    for v in df_3head['product_type']]

train_df = df_3head[df_3head['split']=='train'].reset_index(drop=True)
val_df   = df_3head[df_3head['split']=='val'].reset_index(drop=True)
test_df  = df_3head[df_3head['split']=='test'].reset_index(drop=True)

print(f"Train : {len(train_df):,}  Val : {len(val_df):,}  "
      f"Test : {len(test_df):,}")
print(f"Products ({NUM_PRODUCT_TYPES}) : {all_products}")
print("\n📊 Val defect distribution:")
for i,name in enumerate(SEMANTIC_GROUPS):
    n = (val_df['defect_type_label']==i).sum()
    print(f"   [{i}] {name:<22} {n:>4}")
print("✅ CELL 2 COMPLETE")

Loaded : 21,344 rows
Train : 16,337  Val : 3,339  Test : 1,668
Products (17) : ['bottle', 'cable', 'capsule', 'carpet', 'casting_product', 'grid', 'hazelnut', 'leather', 'magnetic_tile', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']

📊 Val defect distribution:
   [0] contamination            33
   [1] cut                      17
   [2] deformation              15
   [3] fracture                 43
   [4] hole_void                54
   [5] minor_defect             42
   [6] scratch                  29
   [7] surface_quality          63
✅ CELL 2 COMPLETE


In [3]:
# ======================================================================
# CELL 3 — DATASET + DATALOADERS
# ======================================================================
class ThreeHeadDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'])
        if img is None:
            img = np.zeros((224,224,3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        if self.transform: img = self.transform(img)
        return (img,
                int(row['binary_label']),
                int(row['defect_type_label']),
                int(row['product_type_label']),
                row['image_path'])

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3,0.3,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = ThreeHeadDataset(train_df, train_transform)
val_ds   = ThreeHeadDataset(val_df,   val_transform)
test_ds  = ThreeHeadDataset(test_df,  val_transform)

# ── Weighted sampler ──────────────────────────────────────────────────
def make_weighted_sampler(df):
    counts = np.array([max(1,(df['defect_type_label']==i).sum())
                       for i in range(NUM_DEFECT_TYPES)], dtype=float)
    cw = 1.0/counts; cw = cw/cw.sum()
    weights = [0.3 if int(row['defect_type_label'])==-1
               else float(cw[int(row['defect_type_label'])])
               for _,row in df.iterrows()]
    return WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.float32),
        len(weights), replacement=True)

# ── CutMix collate ────────────────────────────────────────────────────
def cutmix_collate(batch):
    imgs,bin_l,def_l,prod_l,paths = zip(*batch)
    imgs  = torch.stack(imgs)
    bin_l = torch.tensor(bin_l,  dtype=torch.long)
    def_l = torch.tensor(def_l,  dtype=torch.long)
    prod_l= torch.tensor(prod_l, dtype=torch.long)
    known = (def_l!=-1).nonzero(as_tuple=True)[0]
    if len(known)>=4 and torch.rand(1).item()<0.5:
        lam  = float(np.clip(np.random.beta(1.,1.), 0.3, 0.7))
        perm = known[torch.randperm(len(known))]
        B,C,H,W = imgs[known].shape
        cut_h = int(H*np.sqrt(1-lam)); cut_w = int(W*np.sqrt(1-lam))
        cx=np.random.randint(W); cy=np.random.randint(H)
        x1=max(0,cx-cut_w//2); x2=min(W,cx+cut_w//2)
        y1=max(0,cy-cut_h//2); y2=min(H,cy+cut_h//2)
        imgs_c=imgs.clone()
        imgs_c[known,:,y1:y2,x1:x2]=imgs[perm,:,y1:y2,x1:x2]
        imgs=imgs_c
        if 1-(x2-x1)*(y2-y1)/(H*W) < 0.5:
            dl2=def_l.clone(); dl2[known]=def_l[perm]; def_l=dl2
    return imgs,bin_l,def_l,prod_l,list(paths)

BATCH_SIZE  = 32
NUM_WORKERS = 4
sampler = make_weighted_sampler(train_df)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
    drop_last=True, collate_fn=cutmix_collate)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")
print("✅ CELL 3 COMPLETE")

Train batches : 510
Val   batches : 105
Test  batches : 53
✅ CELL 3 COMPLETE


In [11]:
# ======================================================================
# CELL 4 FINAL — EdgeNetV3 with Identity-initialized CoordAttention
# conv_h and conv_w biases start at +10 → sigmoid(10)≈1.0 → identity
# This preserves teacher features perfectly at epoch 0
# ======================================================================

class CoordAttention(nn.Module):
    def __init__(self, channels, reduction=32):
        super().__init__()
        mid = max(8, channels // reduction)
        self.conv1  = nn.Conv2d(channels, mid, 1, bias=False)
        self.bn1    = nn.BatchNorm2d(mid)
        self.act    = nn.ReLU()
        # bias=True so we can initialise to identity
        self.conv_h = nn.Conv2d(mid, channels, 1, bias=True)
        self.conv_w = nn.Conv2d(mid, channels, 1, bias=True)

        # ── Identity initialisation ────────────────────────────────────
        # conv_h/conv_w weights → 0, bias → +10
        # sigmoid(0*x + 10) ≈ 0.99997 ≈ 1.0  →  x*1.0*1.0 = x (identity)
        # As training proceeds, bias drifts away from 10 and CA activates
        nn.init.zeros_(self.conv_h.weight)
        nn.init.constant_(self.conv_h.bias, 10.0)
        nn.init.zeros_(self.conv_w.weight)
        nn.init.constant_(self.conv_w.bias, 10.0)

    def forward(self, x):
        B, C, H, W = x.shape
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0,1,3,2)
        y   = torch.cat([x_h, x_w], dim=2)
        y   = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [H, W], dim=2)
        x_w = x_w.permute(0,1,3,2)
        return x * torch.sigmoid(self.conv_h(x_h)) \
                 * torch.sigmoid(self.conv_w(x_w))


class EdgeNetV3(nn.Module):
    def __init__(self, num_defect=8, num_product=17):
        super().__init__()
        base = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

        self.early_features = base.features[:5]
        self.coord_att1     = CoordAttention(80)
        self.late_features  = base.features[5:]
        self.coord_att2     = CoordAttention(1280)
        self.pool           = nn.AdaptiveAvgPool2d(1)

        self.shared = nn.Sequential(
            nn.Linear(1280, 512),
            nn.SiLU(),
            nn.Dropout(0.3),
        )
        self.binary_head = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )
        self.defect_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_defect),
        )
        self.product_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Linear(256, num_product),
        )

    def freeze_backbone(self, freeze=True):
        for p in self.early_features.parameters():
            p.requires_grad = not freeze
        for p in self.late_features.parameters():
            p.requires_grad = not freeze

    def count_params(self):
        return sum(p.numel() for p in self.parameters()) / 1e6

    def forward(self, x):
        x = self.early_features(x)
        x = self.coord_att1(x)
        x = self.late_features(x)
        x = self.coord_att2(x)
        x = self.pool(x).flatten(1)
        x = self.shared(x)
        return self.binary_head(x), \
               self.defect_head(x), \
               self.product_head(x)


# ── Build ─────────────────────────────────────────────────────────────
model = EdgeNetV3(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)

# Quick sanity — CA should be near-identity at init
with torch.no_grad():
    dummy = torch.ones(1, 80, 7, 7).to(device)
    out   = model.coord_att1(dummy)
    ratio = (out / dummy).mean().item()
    print(f"CA identity check: output/input ratio = {ratio:.6f}  "
          f"(should be ~1.0)")

print(f"✅ CELL 4 FINAL — EdgeNetV3 built")
print(f"   Params : {model.count_params():.2f}M")
print(f"   CoordAttention initialised as identity ✅")

CA identity check: output/input ratio = 0.999909  (should be ~1.0)
✅ CELL 4 FINAL — EdgeNetV3 built
   Params : 5.16M
   CoordAttention initialised as identity ✅


In [6]:
# ======================================================================
# CELL 5 — CRITERION DICT + LOSS FUNCTIONS
# ======================================================================

class FocalLoss(nn.Module):
    def __init__(self, gamma_per_class, weight=None, ignore_index=-1):
        super().__init__()
        self.gamma        = torch.tensor(gamma_per_class,
                                         dtype=torch.float32)
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        if valid.sum() == 0:
            return logits.sum() * 0.0
        logits  = logits[valid]; targets = targets[valid]
        gamma   = self.gamma.to(logits.device)
        ce      = F.cross_entropy(
            logits, targets,
            weight=self.weight, reduction='none')
        pt      = torch.exp(-ce)
        g       = gamma[targets]
        return ((1 - pt)**g * ce).mean()


class HomoscedasticLoss(nn.Module):
    def __init__(self, n_tasks=3):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(n_tasks))

    def weights(self):
        return torch.exp(-self.log_vars).detach()

    def set_epoch(self, epoch):
        pass   # API compatibility

    def forward(self, losses):
        w = torch.exp(-self.log_vars)
        return sum(w[i]*losses[i] + self.log_vars[i]
                   for i in range(len(losses)))


# ── Class weights ─────────────────────────────────────────────────────
counts = torch.tensor(
    [max(1,(train_df['defect_type_label']==i).sum())
     for i in range(NUM_DEFECT_TYPES)], dtype=torch.float32)
defect_weights = (1.0/counts / (1.0/counts).sum() * NUM_DEFECT_TYPES
                  ).to(device)

# Higher gamma for hard classes
GAMMA_PER_CLASS = [
    2.0,   # contamination
    2.5,   # cut
    4.0,   # deformation  ← hardest
    2.0,   # fracture
    2.0,   # hole_void
    2.0,   # minor_defect
    4.0,   # scratch       ← hard
    2.0,   # surface_quality
]

n_def  = (train_df['binary_label']==1).sum()
n_norm = (train_df['binary_label']==0).sum()
pos_weight = torch.tensor(
    [n_norm / max(n_def,1)], dtype=torch.float32).to(device)

criterion_dict = {
    'binary':    nn.BCEWithLogitsLoss(pos_weight=pos_weight),
    'defect':    FocalLoss(GAMMA_PER_CLASS, defect_weights, -1),
    'product':   nn.CrossEntropyLoss(),
    'multitask': HomoscedasticLoss(3).to(device),
}

print("✅ CELL 5 COMPLETE — criterion_dict built")
print(f"   pos_weight       : {pos_weight.item():.3f}")
print(f"   defect_weights   : "
      f"{[f'{w:.3f}' for w in defect_weights.cpu()]}")
print(f"   gamma_per_class  : {GAMMA_PER_CLASS}")
print(f"   task weights     : {criterion_dict['multitask'].weights()}")

✅ CELL 5 COMPLETE — criterion_dict built
   pos_weight       : 0.706
   defect_weights   : ['0.601', '0.893', '0.459', '0.738', '1.039', '2.756', '0.512', '1.002']
   gamma_per_class  : [2.0, 2.5, 4.0, 2.0, 2.0, 2.0, 4.0, 2.0]
   task weights     : tensor([1., 1., 1.], device='cuda:0')


In [12]:
# ======================================================================
# V3_TRANSFER_FINAL — Transfer + verify  (run after CELL 4 FINAL)
# ======================================================================
from sklearn.metrics import precision_recall_fscore_support
import numpy as np

# ── Load teacher into temp wrapper ────────────────────────────────────
class EfficientNetBase(nn.Module):
    def __init__(self, num_defect=8, num_product=17):
        super().__init__()
        base = models.efficientnet_b0(weights=None)
        self.features = base.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.shared = nn.Sequential(
            nn.Linear(1280,512), nn.SiLU(), nn.Dropout(0.3))
        self.binary_head = nn.Sequential(
            nn.Linear(512,128), nn.ReLU(), nn.Linear(128,1))
        self.defect_head = nn.Sequential(
            nn.Linear(512,256), nn.SiLU(),
            nn.Dropout(0.2), nn.Linear(256,num_defect))
        self.product_head = nn.Sequential(
            nn.Linear(512,256), nn.SiLU(),
            nn.Linear(256,num_product))
    def forward(self, x):
        x = self.features(x); x = self.pool(x).flatten(1)
        s = self.shared(x)
        return self.binary_head(s), self.defect_head(s), self.product_head(s)

ck_t = torch.load(
    '/home/sufi/training_results/baseline_models/efficientnet_b0_best.pth',
    map_location=device, weights_only=False)

tmp = EfficientNetBase(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)
m, u = tmp.load_state_dict(ck_t, strict=False)
print(f"Temp model: missing={len(m)}  unexpected={len(u)}")

# ── Transfer backbone modules directly ───────────────────────────────
m1, u1 = model.early_features.load_state_dict(
    tmp.features[:5].state_dict(), strict=True)
m2, u2 = model.late_features.load_state_dict(
    tmp.features[5:].state_dict(), strict=True)
model.shared.load_state_dict(
    tmp.shared.state_dict(), strict=True)
model.binary_head.load_state_dict(
    tmp.binary_head.state_dict(), strict=True)
model.defect_head.load_state_dict(
    tmp.defect_head.state_dict(), strict=True)
model.product_head.load_state_dict(
    tmp.product_head.state_dict(), strict=True)

print(f"early_features : missing={len(m1)} unexpected={len(u1)}")
print(f"late_features  : missing={len(m2)} unexpected={len(u2)}")
print(f"heads + shared : ✅")

del tmp
import gc; gc.collect()
torch.cuda.empty_cache()

# ── Evaluator ─────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_pred, all_true = [], []
    bin_c = prod_c = tot = def_c = def_tot = 0
    with torch.no_grad():
        for batch in loader:
            imgs,bin_lbl,def_lbl,prod_lbl,_ = batch
            imgs=imgs.to(device); bin_lbl=bin_lbl.to(device)
            def_lbl=def_lbl.to(device); prod_lbl=prod_lbl.to(device)
            sb,sd,sp = model(imgs)
            bin_c  +=((torch.sigmoid(sb.squeeze())>0.5).long()
                      ==bin_lbl).sum().item()
            prod_c +=(sp.argmax(1)==prod_lbl).sum().item()
            tot    +=imgs.size(0)
            known   = def_lbl!=-1
            if known.sum()>0:
                preds    = sd[known].argmax(1)
                def_c   +=(preds==def_lbl[known]).sum().item()
                def_tot +=known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())
    b=100.*bin_c/max(tot,1); d=100.*def_c/max(def_tot,1)
    p2=100.*prod_c/max(tot,1)
    p,r,f,_=precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return b, d, p2, float(f.mean()), p, r, f

# ── Verify ────────────────────────────────────────────────────────────
print(f"\nRunning val verification...")
v_bin,v_def,v_prod,v_f1,vp,vr,vf = evaluate(model, val_loader)

print(f"\nVal BEFORE fine-tuning:")
print(f"  Bin={v_bin:.1f}%  Def={v_def:.1f}%  "
      f"F1={v_f1:.3f}  Prod={v_prod:.1f}%")
print()
print(f"  {'Class':<22} {'P':>6}  {'R':>6}  {'F1':>6}")
print(f"  {'-'*40}")
for i,name in enumerate(SEMANTIC_GROUPS):
    flag = "  ← ❌ DEAD" if vr[i]<0.1 else ""
    print(f"  {name:<22} {vp[i]:>6.3f}  "
          f"{vr[i]:>6.3f}  {vf[i]:>6.3f}{flag}")
print(f"  {'-'*40}")
print(f"  {'MACRO':<22} {float(np.mean(vp)):>6.3f}  "
      f"{float(np.mean(vr)):>6.3f}  {v_f1:>6.3f}")

if v_def >= 88.0:
    print(f"\n  ✅ SUCCESS — Def={v_def:.1f}%  ready for CELL 6")
else:
    print(f"\n  ❌ Def={v_def:.1f}% — identity init didn't work")
    print(f"  CA ratio check:")
    with torch.no_grad():
        d = torch.ones(1,80,7,7).to(device)
        r = (model.coord_att1(d)/d).mean().item()
        print(f"  coord_att1 ratio={r:.6f}  (must be ~1.0)")

print("✅ V3_TRANSFER_FINAL COMPLETE")

Temp model: missing=0  unexpected=0
early_features : missing=0 unexpected=0
late_features  : missing=0 unexpected=0
heads + shared : ✅

Running val verification...

Val BEFORE fine-tuning:
  Bin=98.9%  Def=90.9%  F1=0.892  Prod=99.2%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.938   0.909   0.923
  cut                     1.000   0.765   0.867
  deformation             0.800   0.800   0.800
  fracture                0.894   0.977   0.933
  hole_void               0.963   0.963   0.963
  minor_defect            0.745   0.905   0.817
  scratch                 1.000   0.793   0.885
  surface_quality         0.967   0.937   0.952
  ----------------------------------------
  MACRO                   0.913   0.881   0.892

  ✅ SUCCESS — Def=90.9%  ready for CELL 6
✅ V3_TRANSFER_FINAL COMPLETE


In [14]:
# ======================================================================
# CELL 6 — Fine-tune EdgeNetV3  (30 epochs from teacher-initialised weights)
# coord_att: LR=1e-4  (new layers, need to learn fast)
# backbone:  LR=2e-5  (already strong, small nudge only)
# heads:     LR=2e-5  (already strong, small nudge only)
# Expected:  beat teacher 93.92% by epoch 5-10
# ======================================================================
from pathlib import Path

V3_DIR  = Path('/home/sufi/training_results/models/V3')
V3_DIR.mkdir(parents=True, exist_ok=True)
V3_SAVE = V3_DIR / 'EdgeNet_V3_best.pth'

EPOCHS = 30

# ── Optimizer ─────────────────────────────────────────────────────────
bb_params    = list(model.early_features.parameters()) + \
               list(model.late_features.parameters())
coord_params = list(model.coord_att1.parameters()) + \
               list(model.coord_att2.parameters())
head_params  = list(model.shared.parameters())       + \
               list(model.binary_head.parameters())  + \
               list(model.defect_head.parameters())  + \
               list(model.product_head.parameters()) + \
               list(criterion_dict['multitask'].parameters())

optimizer = torch.optim.AdamW([
    {'params': bb_params,    'lr': 2e-5},   # pretrained — low LR
    {'params': coord_params, 'lr': 1e-4},   # new layers — higher LR
    {'params': head_params,  'lr': 2e-5},   # pretrained — low LR
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = GradScaler('cuda')

model.freeze_backbone(False)   # everything trainable from epoch 1
best_f1   = 0.0

# ── Evaluator ─────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_pred, all_true = [], []
    bin_c = prod_c = tot = def_c = def_tot = 0
    with torch.no_grad():
        for batch in loader:
            imgs,bin_lbl,def_lbl,prod_lbl,_ = batch
            imgs=imgs.to(device); bin_lbl=bin_lbl.to(device)
            def_lbl=def_lbl.to(device); prod_lbl=prod_lbl.to(device)
            sb,sd,sp = model(imgs)
            bin_c  +=((torch.sigmoid(sb.squeeze())>0.5).long()
                      ==bin_lbl).sum().item()
            prod_c +=(sp.argmax(1)==prod_lbl).sum().item()
            tot    +=imgs.size(0)
            known   = def_lbl!=-1
            if known.sum()>0:
                preds    = sd[known].argmax(1)
                def_c   +=(preds==def_lbl[known]).sum().item()
                def_tot +=known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())
    bin_acc  = 100.*bin_c/max(tot,1)
    def_acc  = 100.*def_c/max(def_tot,1)
    prod_acc = 100.*prod_c/max(tot,1)
    p,r,f,_ = precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return bin_acc, def_acc, prod_acc, float(f.mean()), p, r, f

# ── Training loop ─────────────────────────────────────────────────────
for epoch in range(1, EPOCHS+1):

    criterion_dict['multitask'].set_epoch(epoch)
    model.train()

    tr_bin_c=tr_def_c=tr_prod_c=tr_def_tot=tr_tot=0
    tr_pred, tr_true = [], []

    for batch in train_loader:
        imgs,bin_lbl,def_lbl,prod_lbl,_ = batch
        imgs=imgs.to(device); bin_lbl=bin_lbl.to(device)
        def_lbl=def_lbl.to(device); prod_lbl=prod_lbl.to(device)

        optimizer.zero_grad()

        with autocast('cuda'):
            sb, sd, sp = model(imgs)
            sb_sq = sb.squeeze()
            known = def_lbl != -1

            hard_bin  = criterion_dict['binary'](sb_sq, bin_lbl.float())
            hard_prod = criterion_dict['product'](sp, prod_lbl)
            hard_def  = criterion_dict['defect'](
                sd[known], def_lbl[known]) if known.sum()>0 \
                else torch.tensor(0.0, device=device)

            tw = criterion_dict['multitask'].weights()
            total_loss = (float(tw[0])*hard_bin  +
                          float(tw[1])*hard_def   +
                          float(tw[2])*hard_prod)

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        tr_tot    += imgs.size(0)
        tr_bin_c  +=((torch.sigmoid(sb_sq.detach())>0.5).long()
                     ==bin_lbl).sum().item()
        tr_prod_c +=(sp.detach().argmax(1)==prod_lbl).sum().item()
        if known.sum()>0:
            preds = sd.detach()[known].argmax(1)
            tr_def_c   +=(preds==def_lbl[known]).sum().item()
            tr_def_tot +=known.sum().item()
            tr_pred.extend(preds.cpu().tolist())
            tr_true.extend(def_lbl[known].cpu().tolist())

    scheduler.step()

    # ── Metrics ───────────────────────────────────────────────────────
    t_bin  = 100.*tr_bin_c/max(tr_tot,1)
    t_def  = 100.*tr_def_c/max(tr_def_tot,1)
    t_prod = 100.*tr_prod_c/max(tr_tot,1)
    if tr_true:
        _,_,tf,_ = precision_recall_fscore_support(
            tr_true, tr_pred,
            labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
        t_f1 = float(tf.mean())
    else:
        t_f1 = 0.0

    v_bin,v_def,v_prod,v_f1,vp,vr,vf = evaluate(model, val_loader)
    cur_lr = optimizer.param_groups[1]['lr']   # coord_att LR
    tw     = criterion_dict['multitask'].weights()

    # ── Print ─────────────────────────────────────────────────────────
    beat = "  🏆 BEATS TEACHER" if v_def > 93.92 else ""
    print(f"\nEpoch [{epoch:>2}/{EPOCHS}]  LR={cur_lr:.1e}  "
          f"TaskW=[bin={float(tw[0]):.2f} "
          f"def={float(tw[1]):.2f} "
          f"prod={float(tw[2]):.2f}]  [FINETUNE]{beat}")
    print()
    print(f"  Train → Bin={t_bin:.1f}%  Def={t_def:.1f}%  "
          f"F1={t_f1:.3f}  Prod={t_prod:.1f}%")
    print(f"  Val   → Bin={v_bin:.1f}%  Def={v_def:.1f}%  "
          f"F1={v_f1:.3f}  Prod={v_prod:.1f}%")
    print()
    print(f"  {'Class':<22} {'P':>6}  {'R':>6}  {'F1':>6}")
    print(f"  {'-'*40}")
    for i,name in enumerate(SEMANTIC_GROUPS):
        print(f"  {name:<22} {vp[i]:>6.3f}  {vr[i]:>6.3f}  {vf[i]:>6.3f}")
    print(f"  {'-'*40}")
    print(f"  {'MACRO':<22} {float(np.mean(vp)):>6.3f}  "
          f"{float(np.mean(vr)):>6.3f}  {v_f1:>6.3f}")

    # ── Save best ──────────────────────────────────────────────────────
    if v_f1 > best_f1:
        best_f1 = v_f1
        torch.save({
            'epoch':               epoch,
            'model_state_dict':    model.state_dict(),
            'multitask_state':     criterion_dict['multitask'].state_dict(),
            'optimizer_state':     optimizer.state_dict(),
            'val_score':           v_f1,
            'val_metrics': {
                'binary_acc':      v_bin,
                'defect_acc':      v_def,
                'defect_f1_macro': v_f1,
                'product_acc':     v_prod,
            },
            'defect_type_to_idx':  DEFECT_TYPE_TO_IDX,
            'product_type_to_idx': PRODUCT_TYPE_TO_IDX,
            'idx_to_defect_type':  IDX_TO_DEFECT_TYPE,
            'idx_to_product_type': IDX_TO_PRODUCT_TYPE,
        }, V3_SAVE)
        print(f"\n  ✅ NEW BEST  F1={best_f1:.3f} "
              f"→ saved V3/EdgeNet_V3_best.pth")

print(f"\n{'='*60}")
print(f"EdgeNetV3 TRAINING COMPLETE")
print(f"  Best Val F1  : {best_f1:.3f}")
print(f"  Saved to     : {V3_SAVE}")
print(f"{'='*60}")
print("✅ CELL 6 COMPLETE")


Epoch [ 1/30]  LR=1.0e-04  TaskW=[bin=1.00 def=1.00 prod=1.00]  [FINETUNE]

  Train → Bin=99.3%  Def=79.8%  F1=0.798  Prod=97.3%
  Val   → Bin=99.0%  Def=89.2%  F1=0.844  Prod=99.9%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.968   0.909   0.938
  cut                     0.938   0.882   0.909
  deformation             0.833   0.333   0.476
  fracture                0.811   1.000   0.896
  hole_void               0.964   0.981   0.972
  minor_defect            0.830   0.929   0.876
  scratch                 1.000   0.621   0.766
  surface_quality         0.871   0.968   0.917
  ----------------------------------------
  MACRO                   0.902   0.828   0.844

  ✅ NEW BEST  F1=0.844 → saved V3/EdgeNet_V3_best.pth

Epoch [ 2/30]  LR=9.9e-05  TaskW=[bin=1.00 def=1.00 prod=1.00]  [FINETUNE]

  Train → Bin=99.3%  Def=82.2%  F1=0.820  Prod=97.3%
  Val   → Bin=99.0%  Def=88.5%  F1=0.848  Prod=100.0%

  Class   

In [16]:
# ======================================================================
# CELL 7 — FINAL TEST EVALUATION + THESIS TABLE
# EdgeNetV3 vs Teacher vs all baselines
# ======================================================================
print("Loading best EdgeNetV3 checkpoint...")
ck_v3 = torch.load(V3_SAVE, map_location=device, weights_only=False)
model.load_state_dict(ck_v3['model_state_dict'])
model.eval()
print(f"Loaded epoch {ck_v3['epoch']}  "
      f"val F1={ck_v3['val_metrics']['defect_f1_macro']:.3f}")

# ── Load teacher for comparison ───────────────────────────────────────
class EfficientNetTeacher(nn.Module):
    def __init__(self, num_defect=8, num_product=17):
        super().__init__()
        base = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.shared = nn.Sequential(
            nn.Linear(1280,512), nn.SiLU(), nn.Dropout(0.3))
        self.binary_head = nn.Sequential(
            nn.Linear(512,128), nn.ReLU(), nn.Linear(128,1))
        self.defect_head = nn.Sequential(
            nn.Linear(512,256), nn.SiLU(),
            nn.Dropout(0.2), nn.Linear(256,num_defect))
        self.product_head = nn.Sequential(
            nn.Linear(512,256), nn.SiLU(), nn.Linear(256,num_product))
    def forward(self, x):
        x = self.features(x); x = self.pool(x).flatten(1)
        s = self.shared(x)
        return self.binary_head(s), self.defect_head(s), \
               self.product_head(s)

teacher = EfficientNetTeacher(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)
ck_t = torch.load(
    '/home/sufi/training_results/baseline_models/efficientnet_b0_best.pth',
    map_location=device, weights_only=False)
teacher.load_state_dict(ck_t, strict=False)
teacher.eval()
for p in teacher.parameters(): p.requires_grad = False

# Add this to CELL 7 after loading V3
TTA_AUGS = [
    lambda x: x,
    lambda x: torch.flip(x, [3]),
    lambda x: torch.flip(x, [2]),
    lambda x: torch.rot90(x, 1, [2,3]),
    lambda x: torch.rot90(x, 3, [2,3]),
]

def evaluate_tta(model, loader):
    model.eval()
    all_pred, all_true = [], []
    bin_c=prod_c=tot=def_c=def_tot=0
    with torch.no_grad():
        for batch in loader:
            imgs,bin_lbl,def_lbl,prod_lbl,_ = batch
            imgs=imgs.to(device)
            bin_lbl=bin_lbl.to(device)
            def_lbl=def_lbl.to(device)
            prod_lbl=prod_lbl.to(device)
            B = imgs.size(0)
            avg_bin  = torch.zeros(B, device=device)
            avg_def  = torch.zeros(B, NUM_DEFECT_TYPES, device=device)
            avg_prod = torch.zeros(B, NUM_PRODUCT_TYPES, device=device)
            for aug in TTA_AUGS:
                sb,sd,sp = model(aug(imgs))
                avg_bin  += torch.sigmoid(sb.squeeze(1)) / len(TTA_AUGS)
                avg_def  += torch.softmax(sd, dim=1)     / len(TTA_AUGS)
                avg_prod += torch.softmax(sp, dim=1)     / len(TTA_AUGS)
            bin_c  +=((avg_bin>0.5).long()==bin_lbl).sum().item()
            prod_c +=(avg_prod.argmax(1)==prod_lbl).sum().item()
            tot    +=imgs.size(0)
            known   = def_lbl!=-1
            if known.sum()>0:
                preds    = avg_def[known].argmax(1)
                def_c   +=(preds==def_lbl[known]).sum().item()
                def_tot +=known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())
    b=100.*bin_c/max(tot,1); d=100.*def_c/max(def_tot,1)
    p2=100.*prod_c/max(tot,1)
    p,r,f,s = precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return b, d, p2, float(f.mean()), p, r, f, s

# ── Evaluator ─────────────────────────────────────────────────────────
def test_eval(mdl, loader, name):
    mdl.eval()
    all_pred,all_true=[],[]
    bin_c=prod_c=tot=def_c=def_tot=0
    with torch.no_grad():
        for batch in loader:
            imgs,bin_lbl,def_lbl,prod_lbl,_ = batch
            imgs=imgs.to(device); bin_lbl=bin_lbl.to(device)
            def_lbl=def_lbl.to(device); prod_lbl=prod_lbl.to(device)
            sb,sd,sp = mdl(imgs)
            bin_c  +=((torch.sigmoid(sb.squeeze())>0.5).long()
                      ==bin_lbl).sum().item()
            prod_c +=(sp.argmax(1)==prod_lbl).sum().item()
            tot    +=imgs.size(0)
            known   = def_lbl!=-1
            if known.sum()>0:
                preds    = sd[known].argmax(1)
                def_c   +=(preds==def_lbl[known]).sum().item()
                def_tot +=known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())
    b=100.*bin_c/max(tot,1); d=100.*def_c/max(def_tot,1)
    p2=100.*prod_c/max(tot,1)
    p,r,f,s = precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return dict(name=name,bin=b,defect=d,prod=p2,
                f1=float(f.mean()),p=p,r=r,f=f,s=s)

print("\nEvaluating on test set...")
r_v3  = test_eval(model,   test_loader, "EdgeNetV3 (Ours)")
r_tea = test_eval(teacher, test_loader, "EfficientNet-B0")

# ── Table ─────────────────────────────────────────────────────────────
print()
print("="*70)
print("FINAL TEST SET — THESIS TABLE")
print("="*70)
hdr = (f"  {'Model':<28} {'Params':>7}  "
       f"{'Binary':>8}  {'Defect':>8}  {'F1':>7}  {'Product':>9}")
div = "  "+"-"*66
print(hdr); print(div)

for name,defect in [
    ("MobileNetV3-Small",  "79.05%"),
    ("ResNet-18",          "84.46%"),
    ("ResNet-50",          "90.88%"),
    ("MobileNetV3-Large",  "91.22%"),
    ("EdgeNetV2+ENH",      "~89.2%"),
]:
    print(f"  {name:<28} {'—':>7}  {'—':>8}  "
          f"{defect:>8}  {'—':>7}  {'—':>9}")

print(div)
params_t = sum(p.numel() for p in teacher.parameters())/1e6
print(f"  {r_tea['name']:<28} {params_t:>6.2f}M  "
      f"{r_tea['bin']:>7.2f}%  {r_tea['defect']:>7.2f}%  "
      f"{r_tea['f1']:>7.3f}  {r_tea['prod']:>8.2f}%  ← Baseline Teacher")
print(div)
params_v3 = model.count_params()
beat = "  ✅ BEATS TEACHER" if r_v3['defect'] > r_tea['defect'] else \
       "  ⚠️  Close"
print(f"  {r_v3['name']:<28} {params_v3:>6.2f}M  "
      f"{r_v3['bin']:>7.2f}%  {r_v3['defect']:>7.2f}%  "
      f"{r_v3['f1']:>7.3f}  {r_v3['prod']:>8.2f}%{beat}")
print("="*70)

# ── Per-class ─────────────────────────────────────────────────────────
print()
print("="*70)
print("PER-CLASS F1 — EdgeNetV3 vs EfficientNet-B0  (Test Set)")
print("="*70)
print(f"  {'Class':<22}  {'Teacher':>9}  {'EdgeNetV3':>10}  "
      f"{'Gain':>8}  {'Support':>8}")
print("  "+"-"*60)
for i,name in enumerate(SEMANTIC_GROUPS):
    ft = r_tea['f'][i]; fv = r_v3['f'][i]
    g  = fv - ft
    flag = "  ▲" if g>0.02 else ("  ▼" if g<-0.02 else "")
    print(f"  {name:<22}  {ft:>9.3f}  {fv:>10.3f}  "
          f"{g:>+8.3f}  {int(r_v3['s'][i]):>8}{flag}")
print("  "+"-"*60)
print(f"  {'MACRO':<22}  {r_tea['f1']:>9.3f}  "
      f"{r_v3['f1']:>10.3f}  "
      f"{r_v3['f1']-r_tea['f1']:>+8.3f}")
print("="*70)

print(f"\n  Teacher defect acc : {r_tea['defect']:.2f}%")
print(f"  EdgeNetV3 defect   : {r_v3['defect']:.2f}%")
if r_v3['defect'] > r_tea['defect']:
    print(f"  ✅ EdgeNetV3 BEATS teacher by "
          f"+{r_v3['defect']-r_tea['defect']:.2f}%")
else:
    print(f"  Gap to teacher : "
          f"{r_tea['defect']-r_v3['defect']:.2f}%")

print("\n✅ CELL 7 COMPLETE — EdgeNetV3 evaluation done")

Loading best EdgeNetV3 checkpoint...
Loaded epoch 30  val F1=0.903

Evaluating on test set...

FINAL TEST SET — THESIS TABLE
  Model                         Params    Binary    Defect       F1    Product
  ------------------------------------------------------------------
  MobileNetV3-Small                  —         —    79.05%        —          —
  ResNet-18                          —         —    84.46%        —          —
  ResNet-50                          —         —    90.88%        —          —
  MobileNetV3-Large                  —         —    91.22%        —          —
  EdgeNetV2+ENH                      —         —    ~89.2%        —          —
  ------------------------------------------------------------------
  EfficientNet-B0                5.00M    98.68%    91.22%    0.897     99.76%  ← Baseline Teacher
  ------------------------------------------------------------------
  EdgeNetV3 (Ours)               5.16M    98.86%    89.19%    0.857     99.94%  ⚠️  Close

PER-

In [17]:
# ======================================================================
# CELL 6B — Continue training 10 more epochs from V3 best checkpoint
# ======================================================================
from pathlib import Path

V3_SAVE = Path('/home/sufi/training_results/models/V3/EdgeNet_V3_best.pth')

ck_v3 = torch.load(V3_SAVE, map_location=device, weights_only=False)
model.load_state_dict(ck_v3['model_state_dict'])
model.freeze_backbone(False)
print(f"Loaded epoch {ck_v3['epoch']}  "
      f"val F1={ck_v3['val_metrics']['defect_f1_macro']:.3f}")

EXTRA_EPOCHS = 10
best_f1      = ck_v3['val_score']

optimizer_ex = torch.optim.AdamW([
    {'params': list(model.early_features.parameters()) +
               list(model.late_features.parameters()),  'lr': 5e-6},
    {'params': list(model.coord_att1.parameters()) +
               list(model.coord_att2.parameters()),     'lr': 2e-5},
    {'params': list(model.shared.parameters())       +
               list(model.binary_head.parameters())  +
               list(model.defect_head.parameters())  +
               list(model.product_head.parameters()) +
               list(criterion_dict['multitask'].parameters()), 'lr': 5e-6},
], weight_decay=1e-4)

scheduler_ex = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_ex, T_max=EXTRA_EPOCHS, eta_min=1e-7)
scaler_ex    = GradScaler('cuda')

for epoch in range(1, EXTRA_EPOCHS+1):

    criterion_dict['multitask'].set_epoch(epoch)
    model.train()
    tr_bin_c=tr_def_c=tr_prod_c=tr_def_tot=tr_tot=0
    tr_pred,tr_true=[],[]

    for batch in train_loader:
        imgs,bin_lbl,def_lbl,prod_lbl,_ = batch
        imgs=imgs.to(device); bin_lbl=bin_lbl.to(device)
        def_lbl=def_lbl.to(device); prod_lbl=prod_lbl.to(device)
        optimizer_ex.zero_grad()
        with autocast('cuda'):
            sb,sd,sp = model(imgs)
            sb_sq = sb.squeeze(); known = def_lbl!=-1
            hard_bin  = criterion_dict['binary'](sb_sq, bin_lbl.float())
            hard_prod = criterion_dict['product'](sp, prod_lbl)
            hard_def  = criterion_dict['defect'](
                sd[known], def_lbl[known]) if known.sum()>0 \
                else torch.tensor(0.0, device=device)
            tw = criterion_dict['multitask'].weights()
            total_loss = (float(tw[0])*hard_bin +
                          float(tw[1])*hard_def  +
                          float(tw[2])*hard_prod)
        scaler_ex.scale(total_loss).backward()
        scaler_ex.unscale_(optimizer_ex)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_ex.step(optimizer_ex); scaler_ex.update()

        tr_tot    += imgs.size(0)
        tr_bin_c  +=((torch.sigmoid(sb_sq.detach())>0.5).long()
                     ==bin_lbl).sum().item()
        tr_prod_c +=(sp.detach().argmax(1)==prod_lbl).sum().item()
        if known.sum()>0:
            preds = sd.detach()[known].argmax(1)
            tr_def_c   +=(preds==def_lbl[known]).sum().item()
            tr_def_tot +=known.sum().item()
            tr_pred.extend(preds.cpu().tolist())
            tr_true.extend(def_lbl[known].cpu().tolist())

    scheduler_ex.step()

    t_def = 100.*tr_def_c/max(tr_def_tot,1)
    if tr_true:
        _,_,tf,_ = precision_recall_fscore_support(
            tr_true,tr_pred,
            labels=list(range(NUM_DEFECT_TYPES)),zero_division=0)
        t_f1 = float(tf.mean())
    else:
        t_f1 = 0.0

    v_bin,v_def,v_prod,v_f1,vp,vr,vf = evaluate(model, val_loader)
    cur_lr = optimizer_ex.param_groups[1]['lr']
    beat   = "  🏆 BEATS TEACHER!" if v_def > 92.57 else ""

    print(f"\nEpoch [{30+epoch}/{30+EXTRA_EPOCHS}]  "
          f"LR={cur_lr:.1e}  [CONTINUE]{beat}")
    print(f"  Train → Def={t_def:.1f}%  F1={t_f1:.3f}")
    print(f"  Val   → Bin={v_bin:.1f}%  Def={v_def:.1f}%  "
          f"F1={v_f1:.3f}  Prod={v_prod:.1f}%")
    print()
    print(f"  {'Class':<22} {'P':>6}  {'R':>6}  {'F1':>6}")
    print(f"  {'-'*40}")
    for i,name in enumerate(SEMANTIC_GROUPS):
        print(f"  {name:<22} {vp[i]:>6.3f}  "
              f"{vr[i]:>6.3f}  {vf[i]:>6.3f}")
    print(f"  {'-'*40}")
    print(f"  {'MACRO':<22} {float(np.mean(vp)):>6.3f}  "
          f"{float(np.mean(vr)):>6.3f}  {v_f1:>6.3f}")

    if v_f1 > best_f1:
        best_f1 = v_f1
        torch.save({
            'epoch':               30 + epoch,
            'model_state_dict':    model.state_dict(),
            'multitask_state':     criterion_dict['multitask'].state_dict(),
            'optimizer_state':     optimizer_ex.state_dict(),
            'val_score':           v_f1,
            'val_metrics': {
                'binary_acc':      v_bin,
                'defect_acc':      v_def,
                'defect_f1_macro': v_f1,
                'product_acc':     v_prod,
            },
            'defect_type_to_idx':  DEFECT_TYPE_TO_IDX,
            'product_type_to_idx': PRODUCT_TYPE_TO_IDX,
            'idx_to_defect_type':  IDX_TO_DEFECT_TYPE,
            'idx_to_product_type': IDX_TO_PRODUCT_TYPE,
        }, V3_SAVE)
        print(f"\n  ✅ NEW BEST  F1={best_f1:.3f} → saved")

print(f"\n{'='*55}")
print(f"  Best Val F1  : {best_f1:.3f}")
print(f"  Teacher was  : 92.57% defect  (val)")
print(f"  We are at    : {v_def:.2f}% defect  (val)")
if v_def > 92.57:
    print(f"  🏆 BEATS TEACHER by +{v_def-92.57:.2f}%")
print(f"{'='*55}")
print("✅ CELL 6B COMPLETE")

Loaded epoch 30  val F1=0.903

Epoch [31/40]  LR=2.0e-05  [CONTINUE]
  Train → Def=85.8%  F1=0.856
  Val   → Bin=99.1%  Def=91.6%  F1=0.899  Prod=100.0%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.938   0.909   0.923
  cut                     0.941   0.941   0.941
  deformation             1.000   0.667   0.800
  fracture                0.878   1.000   0.935
  hole_void               0.981   0.981   0.981
  minor_defect            0.826   0.905   0.864
  scratch                 1.000   0.690   0.816
  surface_quality         0.897   0.968   0.931
  ----------------------------------------
  MACRO                   0.933   0.883   0.899

Epoch [32/40]  LR=1.8e-05  [CONTINUE]
  Train → Def=85.4%  F1=0.852
  Val   → Bin=99.1%  Def=90.9%  F1=0.885  Prod=100.0%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.938   0.909   0.923
  cut            

#### extra salty

In [4]:
# ======================================================================
# CELL 4 — EdgeNetV3Plus
# Novel additions on top of EfficientNet-B0:
# 1. Multi-scale feature fusion (early 80ch + late 1280ch pooled together)
# 2. Lightweight channel self-attention on fused embedding
# 3. Gated residual connection in shared projection
# These additions are the "salt" — EfficientNet backbone unchanged
# ======================================================================

class CoordAttention(nn.Module):
    def __init__(self, channels, reduction=32):
        super().__init__()
        mid = max(8, channels // reduction)
        self.conv1  = nn.Conv2d(channels, mid, 1, bias=False)
        self.bn1    = nn.BatchNorm2d(mid)
        self.act    = nn.ReLU()
        self.conv_h = nn.Conv2d(mid, channels, 1, bias=True)
        self.conv_w = nn.Conv2d(mid, channels, 1, bias=True)
        # Identity init
        nn.init.zeros_(self.conv_h.weight)
        nn.init.constant_(self.conv_h.bias, 10.0)
        nn.init.zeros_(self.conv_w.weight)
        nn.init.constant_(self.conv_w.bias, 10.0)

    def forward(self, x):
        B, C, H, W = x.shape
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0,1,3,2)
        y   = torch.cat([x_h, x_w], dim=2)
        y   = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [H, W], dim=2)
        x_w = x_w.permute(0,1,3,2)
        return x * torch.sigmoid(self.conv_h(x_h)) \
                 * torch.sigmoid(self.conv_w(x_w))


class ChannelAttention(nn.Module):
    """
    Lightweight SE-style self-attention on 1D feature vector.
    Applied on the fused embedding AFTER pooling.
    Novel: operates on concatenated multi-scale features.
    """
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(16, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid(),
        )
        # Init: near-identity (all gates open)
        nn.init.constant_(self.fc[2].weight, 1.0 / self.fc[2].weight.shape[1])

    def forward(self, x):
        return x * self.fc(x)


class GatedProjection(nn.Module):
    """
    Gated linear projection: output = W1(x) * sigmoid(W2(x))
    (GLU-style). Better gradient flow than plain linear.
    Novel addition to the shared projection.
    """
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        self.proj  = nn.Linear(in_dim, out_dim)
        self.gate  = nn.Linear(in_dim, out_dim)
        self.drop  = nn.Dropout(dropout)
        self.bn    = nn.LayerNorm(out_dim)

    def forward(self, x):
        return self.drop(self.bn(self.proj(x) * torch.sigmoid(self.gate(x))))


class EdgeNetV3Plus(nn.Module):
    """
    EfficientNet-B0 backbone +
    1. CoordAttention at early (80ch) and late (1280ch) stages
    2. Multi-scale fusion: pool both stages, concatenate (80+1280=1360)
    3. ChannelAttention on fused 1360-dim vector
    4. GatedProjection: 1360 → 512 (replaces plain Linear)
    5. Same 3 heads as teacher (key indices preserved for transfer)
    """
    def __init__(self, num_defect=8, num_product=17):
        super().__init__()
        base = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

        # Backbone split (same as V3)
        self.early_features = base.features[:5]   # out: 80ch
        self.coord_att1     = CoordAttention(80)
        self.late_features  = base.features[5:]   # out: 1280ch
        self.coord_att2     = CoordAttention(1280)

        # Two separate pooling paths
        self.pool_late  = nn.AdaptiveAvgPool2d(1)   # 1280-dim
        self.pool_early = nn.AdaptiveAvgPool2d(1)   # 80-dim

        # Novel: channel attention on fused 1360-dim vector
        self.fuse_attn  = ChannelAttention(1360, reduction=8)

        # Novel: gated projection 1360 → 512
        self.shared_proj = GatedProjection(1360, 512, dropout=0.3)

        # ── Heads: key indices match teacher exactly ──────────────────
        # shared.0 not used (replaced by shared_proj above)
        # but we keep a dummy shared.0 so transfer still works
        self.shared = nn.Sequential(
            nn.Linear(1280, 512),  # loaded from teacher, then bypassed
            nn.SiLU(),
            nn.Dropout(0.3),
        )

        self.binary_head = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )
        self.defect_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_defect),
        )
        self.product_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Linear(256, num_product),
        )

    def freeze_backbone(self, freeze=True):
        for p in self.early_features.parameters():
            p.requires_grad = not freeze
        for p in self.late_features.parameters():
            p.requires_grad = not freeze

    def count_params(self):
        total = sum(p.numel() for p in self.parameters())
        new   = sum(p.numel() for p in self.fuse_attn.parameters()) + \
                sum(p.numel() for p in self.shared_proj.parameters()) + \
                sum(p.numel() for p in self.coord_att1.parameters()) + \
                sum(p.numel() for p in self.coord_att2.parameters())
        return total/1e6, new/1e6

    def forward(self, x):
        # Early features + CoordAtt
        e = self.early_features(x)
        e = self.coord_att1(e)

        # Late features + CoordAtt
        l = self.late_features(e)
        l = self.coord_att2(l)

        # Multi-scale pool
        e_pooled = self.pool_early(e).flatten(1)   # [B, 80]
        l_pooled = self.pool_late(l).flatten(1)    # [B, 1280]

        # Fuse + channel attention + gated projection
        fused = torch.cat([e_pooled, l_pooled], dim=1)  # [B, 1360]
        fused = self.fuse_attn(fused)                    # [B, 1360]
        out   = self.shared_proj(fused)                  # [B, 512]

        return self.binary_head(out), \
               self.defect_head(out), \
               self.product_head(out)


# ── Build + verify ─────────────────────────────────────────────────────
model = EdgeNetV3Plus(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)

with torch.no_grad():
    dummy = torch.randn(2,3,224,224).to(device)
    b,d,p = model(dummy)
    assert b.shape==(2,1)
    assert d.shape==(2,NUM_DEFECT_TYPES)
    assert p.shape==(2,NUM_PRODUCT_TYPES)

total, novel = model.count_params()
print(f"✅ CELL 4 COMPLETE — EdgeNetV3Plus")
print(f"   Total params   : {total:.2f}M")
print(f"   Novel additions: {novel:.3f}M  (CoordAtt + FuseAttn + GatedProj)")
print(f"   EfficientNet   : {total-novel:.2f}M  (unchanged)")

# CA identity check
with torch.no_grad():
    d_ = torch.ones(1,80,7,7).to(device)
    r  = (model.coord_att1(d_)/d_).mean().item()
    print(f"   CA identity    : {r:.6f}  (must be ~1.0)")

✅ CELL 4 COMPLETE — EdgeNetV3Plus
   Total params   : 7.01M
   Novel additions: 2.015M  (CoordAtt + FuseAttn + GatedProj)
   EfficientNet   : 5.00M  (unchanged)
   CA identity    : 0.999909  (must be ~1.0)


In [7]:
# ======================================================================
# TAYLOR_CELL — Parameter importance analysis (Taylor criterion)
# Shows which layers matter most BEFORE deciding what to prune
# Run AFTER training to analyse the trained model
# ======================================================================

def compute_taylor_importance(model, loader, n_batches=20):
    """
    Taylor importance = |gradient × weight|
    Higher = more important to keep
    Lower  = safe to prune
    """
    model.train()
    # Zero existing grads
    for p in model.parameters():
        if p.grad is not None:
            p.grad.zero_()

    # Accumulate gradients over n_batches
    n = 0
    for batch in loader:
        if n >= n_batches:
            break
        imgs,bin_lbl,def_lbl,prod_lbl,_ = batch
        imgs=imgs.to(device); bin_lbl=bin_lbl.to(device)
        def_lbl=def_lbl.to(device); prod_lbl=prod_lbl.to(device)

        sb,sd,sp = model(imgs)
        sb_sq = sb.squeeze(); known = def_lbl!=-1

        hard_bin  = criterion_dict['binary'](sb_sq, bin_lbl.float())
        hard_prod = criterion_dict['product'](sp, prod_lbl)
        hard_def  = criterion_dict['defect'](
            sd[known], def_lbl[known]) if known.sum()>0 \
            else torch.tensor(0.0, device=device)

        tw = criterion_dict['multitask'].weights()
        loss = (float(tw[0])*hard_bin +
                float(tw[1])*hard_def  +
                float(tw[2])*hard_prod)
        loss.backward()
        n += 1

    # Compute importance per named layer
    importance = {}
    for name, param in model.named_parameters():
        if param.grad is not None and param.requires_grad:
            # Taylor criterion: |w * grad|
            imp = (param.data * param.grad).abs().mean().item()
            importance[name] = imp

    model.eval()
    return importance

# ── Run importance analysis ────────────────────────────────────────────
print("Computing Taylor importance...")
imp = compute_taylor_importance(model, train_loader, n_batches=30)

# Sort by importance
sorted_imp = sorted(imp.items(), key=lambda x: x[1], reverse=True)

print(f"\n{'='*65}")
print(f"TOP 15 MOST IMPORTANT PARAMETERS")
print(f"{'='*65}")
for name, val in sorted_imp[:15]:
    bar = '█' * int(val * 5000)
    print(f"  {name:<45} {val:.6f}  {bar[:30]}")

print(f"\n{'='*65}")
print(f"BOTTOM 15 LEAST IMPORTANT (pruning candidates)")
print(f"{'='*65}")
for name, val in sorted_imp[-15:]:
    print(f"  {name:<45} {val:.6f}")

# ── Layer group summary ───────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"IMPORTANCE BY LAYER GROUP")
print(f"{'='*65}")
groups = {
    'early_features': [], 'late_features': [],
    'coord_att1': [],     'coord_att2': [],
    'fuse_attn': [],      'shared_proj': [],
    'shared': [],
    'binary_head': [],    'defect_head': [],
    'product_head': [],
}
for name, val in imp.items():
    for g in groups:
        if name.startswith(g):
            groups[g].append(val)
            break

for g, vals in groups.items():
    if vals:
        avg = sum(vals)/len(vals)
        mx  = max(vals)
        bar = '█' * int(avg * 3000)
        print(f"  {g:<20} avg={avg:.6f}  max={mx:.6f}  "
              f"n={len(vals):>4}  {bar[:25]}")

print(f"\n✅ TAYLOR_CELL COMPLETE")
print(f"   Use this to decide what to prune before fine-tuning")

Computing Taylor importance...

TOP 15 MOST IMPORTANT PARAMETERS
  early_features.1.0.block.2.1.weight           0.615625  ██████████████████████████████
  early_features.2.0.block.3.1.weight           0.268786  ██████████████████████████████
  early_features.1.0.block.0.0.weight           0.183177  ██████████████████████████████
  early_features.3.0.block.3.1.weight           0.173112  ██████████████████████████████
  early_features.2.1.block.3.1.weight           0.165680  ██████████████████████████████
  early_features.4.0.block.3.1.weight           0.116195  ██████████████████████████████
  early_features.0.1.bias                       0.102154  ██████████████████████████████
  early_features.1.0.block.0.1.weight           0.101710  ██████████████████████████████
  early_features.0.1.weight                     0.092587  ██████████████████████████████
  early_features.0.0.weight                     0.092259  ██████████████████████████████
  binary_head.2.bias                         

In [10]:
# ======================================================================
# PRUNE_CELL — Structured pruning of least important channels
# Prunes dropout layers and zeroes out low-importance weights
# Target: reduce params by ~10% without accuracy loss
# ======================================================================
import torch.nn.utils.prune as prune

def apply_magnitude_pruning(model, amount=0.15):
    """
    L1 unstructured pruning on Conv2d and Linear layers.
    amount = fraction of weights to zero out (not remove).
    Zeroed weights still exist but are masked — no architecture change.
    """
    pruned_params = 0
    total_params  = 0

    for name, module in model.named_modules():
        # Only prune backbone layers — protect heads
        if isinstance(module, nn.Conv2d):
            if any(x in name for x in
                   ['early_features', 'late_features']):
                prune.l1_unstructured(module, name='weight',
                                      amount=amount)
                pruned_params += int(module.weight.numel() * amount)
                total_params  += module.weight.numel()

    # Make pruning permanent (remove masks, actual zeros)
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            try:
                prune.remove(module, 'weight')
            except:
                pass

    sparsity = pruned_params / max(total_params, 1) * 100
    print(f"  Pruned {pruned_params:,} / {total_params:,} "
          f"conv weights ({sparsity:.1f}% sparsity)")
    return model


def count_nonzero_params(model):
    total   = sum(p.numel() for p in model.parameters())
    nonzero = sum(p.nonzero().size(0) for p in model.parameters())
    return total/1e6, nonzero/1e6


# ── Evaluate before pruning ───────────────────────────────────────────
print("Before pruning:")
v_bin,v_def,v_prod,v_f1,vp,vr,vf = evaluate(model, val_loader)
print(f"  Def={v_def:.2f}%  F1={v_f1:.3f}")

total_b, nz_b = count_nonzero_params(model)
print(f"  Total params : {total_b:.2f}M  NonZero: {nz_b:.2f}M")

# ── Apply pruning ─────────────────────────────────────────────────────
print(f"\nApplying 15% L1 magnitude pruning to backbone convs...")
model = apply_magnitude_pruning(model, amount=0.15)

# ── Evaluate after pruning ────────────────────────────────────────────
print(f"\nAfter pruning (before fine-tuning):")
v_bin2,v_def2,v_prod2,v_f12,vp2,vr2,vf2 = evaluate(model, val_loader)
print(f"  Def={v_def2:.2f}%  F1={v_f12:.3f}  "
      f"(drop: {v_def-v_def2:.2f}%)")

total_a, nz_a = count_nonzero_params(model)
print(f"  Total params : {total_a:.2f}M  NonZero: {nz_a:.2f}M")
print(f"  Effective reduction: {(1-nz_a/nz_b)*100:.1f}%")

if abs(v_def2 - v_def) < 1.0:
    print(f"\n  ✅ Pruning safe — accuracy drop < 1%")
    print(f"  Fine-tune for 5 epochs to recover")
else:
    print(f"\n  ⚠️  Drop > 1% — reduce pruning amount to 0.10")

print("✅ PRUNE_CELL COMPLETE")

Before pruning:
  Def=22.64%  F1=0.185
  Total params : 7.01M  NonZero: 6.91M

Applying 15% L1 magnitude pruning to backbone convs...
  Pruned 593,406 / 3,956,192 conv weights (15.0% sparsity)

After pruning (before fine-tuning):
  Def=20.61%  F1=0.170  (drop: 2.03%)
  Total params : 7.01M  NonZero: 6.32M
  Effective reduction: 8.6%

  ⚠️  Drop > 1% — reduce pruning amount to 0.10
✅ PRUNE_CELL COMPLETE


In [11]:
# ======================================================================
# V3PLUS_TRANSFER — Transfer teacher into EdgeNetV3Plus
# Backbone + heads: same as V3
# New layers (fuse_attn, shared_proj, coord_att): random/identity init
# ======================================================================

class EfficientNetBase(nn.Module):
    def __init__(self, nd=8, np_=17):
        super().__init__()
        base = models.efficientnet_b0(weights=None)
        self.features = base.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.shared = nn.Sequential(
            nn.Linear(1280,512), nn.SiLU(), nn.Dropout(0.3))
        self.binary_head = nn.Sequential(
            nn.Linear(512,128), nn.ReLU(), nn.Linear(128,1))
        self.defect_head = nn.Sequential(
            nn.Linear(512,256), nn.SiLU(),
            nn.Dropout(0.2), nn.Linear(256,nd))
        self.product_head = nn.Sequential(
            nn.Linear(512,256), nn.SiLU(),
            nn.Linear(256,np_))
    def forward(self, x):
        x=self.features(x); x=self.pool(x).flatten(1)
        s=self.shared(x)
        return self.binary_head(s), self.defect_head(s), \
               self.product_head(s)

ck_t = torch.load(
    '/home/sufi/training_results/baseline_models/efficientnet_b0_best.pth',
    map_location=device, weights_only=False)

tmp = EfficientNetBase(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)
tmp.load_state_dict(ck_t, strict=False)

model.early_features.load_state_dict(
    tmp.features[:5].state_dict(), strict=True)
model.late_features.load_state_dict(
    tmp.features[5:].state_dict(), strict=True)
# shared still loaded (used as fallback, bypassed in forward)
model.shared.load_state_dict(tmp.shared.state_dict(), strict=True)
model.binary_head.load_state_dict(
    tmp.binary_head.state_dict(), strict=True)
model.defect_head.load_state_dict(
    tmp.defect_head.state_dict(), strict=True)
model.product_head.load_state_dict(
    tmp.product_head.state_dict(), strict=True)

# Init shared_proj from teacher's shared weights
# proj maps 1360→512, teacher maps 1280→512
# Copy teacher's shared.0.weight into the last 1280 columns of proj
with torch.no_grad():
    t_w = tmp.shared[0].weight.data   # [512, 1280]
    t_b = tmp.shared[0].bias.data     # [512]
    # proj.weight shape: [512, 1360]
    # Fill last 1280 columns with teacher weights
    # First 80 columns (early features): init small random
    model.shared_proj.proj.weight[:, :80]  = \
        torch.randn(512, 80).to(device) * 0.01
    model.shared_proj.proj.weight[:, 80:]  = t_w
    model.shared_proj.proj.bias            = \
        nn.Parameter(t_b.clone())
    # Gate: init to +2 (sigmoid≈0.88 → near-identity at start)
    model.shared_proj.gate.weight[:, :80]  = \
        torch.randn(512, 80).to(device) * 0.01
    model.shared_proj.gate.weight[:, 80:]  = t_w * 0.1
    nn.init.constant_(model.shared_proj.gate.bias, 2.0)

del tmp
gc.collect(); torch.cuda.empty_cache()

print("✅ V3Plus transfer complete")

# Verify
v_bin,v_def,v_prod,v_f1,vp,vr,vf = evaluate(model, val_loader)
print(f"Val before training: Def={v_def:.1f}%  F1={v_f1:.3f}")
if v_def >= 85.0:
    print(f"✅ Transfer worked — ready for CELL 6")
else:
    print(f"⚠️  Lower than expected but should recover in 2-3 epochs")

✅ V3Plus transfer complete
Val before training: Def=90.9%  F1=0.892
✅ Transfer worked — ready for CELL 6
